# 2. OVERNIGHT GAP TRADING — IN-SAMPLE PARAMETER STABILITY & PLATEAU CHECK
## SYSTEMATIC TRADING APPROACH FOLLOWING ALPHACA FRAMEWORK

Objective: confirm that the *chosen* boundary (`gap_min_trade`, `gap_threshold`) sits on a **wide plateau**, not on an over-fitted cliff.

Strict IS only: `[instrument.start, oos_start)`.

In [1]:
# ===== 02 — In-sample grid + plateau =====
import os, sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_upwards(name, start=None, must_be_dir=True):
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        target = base / name
        if target.is_dir() if must_be_dir else target.exists():
            return base
    raise FileNotFoundError(f"Could not find '{name}' walking up from {here}")

EDGE_DIR    = find_upwards('variants')
utils_path  = str(find_upwards('utils') / 'utils')
engine_path = str(EDGE_DIR)
engine_dir  = str(EDGE_DIR / 'overnight_gap')
for p in (utils_path, engine_path, engine_dir):
    if p not in sys.path:
        sys.path.insert(0, p)

import overnight_gap_engine as eng
import variant_config
from variant_config import load_variant, save_json
importlib.reload(eng)
variant_config.EDGE_DIR = EDGE_DIR

VARIANT    = "v1_spy"
STOP_LIMIT = 0.09

cfg        = load_variant(VARIANT)
eng.set_engine_defaults(**cfg.get("engine_defaults", {}))
RUN_DIR    = cfg["run_dir"]
TICKER     = cfg["instrument"]["ticker"]
DATA_START = cfg["instrument"]["start"]
OOS_START  = cfg["oos_start_ts"]
GAP_MIN    = cfg["params"]["gap_min_trade"]
GAP_THR    = cfg["params"]["gap_threshold"]
GAP_MAX_TF = cfg["params"]["gap_max_tf"]
ENTRY_TF   = cfg["params"]["entry_tf_min"]
MR_FILL    = cfg["params"]["mr_fill_target"]
TF_MULT    = cfg["params"]["tf_target_mult"]
MR_STOP    = cfg["params"]["mr_stop_atr_mult"]
TF_STOP    = cfg["params"]["tf_stop_atr_mult"]
TIME_EXIT  = cfg["params"].get("time_exit", "eod")
SKIP_FRI   = cfg["params"].get("skip_fridays", True)
SLIP_BPS   = cfg["params"]["slip_bps_per_side"]
GAP_MIN_VALUES = cfg["grid"]["gap_min_trade"]
GAP_THR_VALUES = cfg["grid"]["gap_threshold"]

FileNotFoundError: No 'v1_spy.yaml' found under /Users/luyandy/Documents/Alphaca Capital/Alphaca-Capital-Research/research_development/02_strategy_development/notebooks/Overnight_Gap/variants

In [ ]:
# ---- Load & filter to IS ----
intraday_raw = eng.load_data(TICKER, start=DATA_START)
daily_raw    = eng.session_daily(intraday_raw)

n_is = int((daily_raw.index < OOS_START).sum())
daily_is = daily_raw.iloc[:n_is].copy()
is_end_date = daily_is.index[-1]

intraday_dates = intraday_raw.index.normalize().tz_localize(None) if intraday_raw.index.tz is not None \
                 else intraday_raw.index.normalize()
oos_start_dt = pd.Timestamp(OOS_START)
if intraday_raw.index.tz is not None:
    oos_start_dt = oos_start_dt.tz_localize(intraday_raw.index.tz)
intraday_is = intraday_raw[intraday_raw.index < oos_start_dt]

print(f"Strict IS: {daily_is.index.min().date()} → {is_end_date.date()} ({len(daily_is)} sessions)")

In [ ]:
# ---- Helpers ----
def pf(t):
    if len(t) == 0: return 0.0
    p = t["pnl"].values
    g = p[p > 0].sum(); l = -p[p < 0].sum()
    return float(g / l) if l > 0 else np.inf

def evaluate(grid_min, grid_thr):
    t = eng.run_gap_backtest(intraday_is, daily_is,
        gap_min_trade=grid_min, gap_threshold=grid_thr, gap_max_tf=GAP_MAX_TF,
        entry_tf_min=ENTRY_TF, mr_fill_target=MR_FILL, tf_target_mult=TF_MULT,
        mr_stop_atr_mult=MR_STOP, tf_stop_atr_mult=TF_STOP, time_exit=TIME_EXIT,
        skip_fridays=SKIP_FRI, slip_bps_per_side=SLIP_BPS)
    return (pf(t), len(t))

pf_grid = pd.DataFrame(index=GAP_MIN_VALUES, columns=GAP_THR_VALUES, dtype=float)
n_grid  = pd.DataFrame(index=GAP_MIN_VALUES, columns=GAP_THR_VALUES, dtype=int)
for gm in GAP_MIN_VALUES:
    for gt in GAP_THR_VALUES:
        p, n = evaluate(gm, gt)
        pf_grid.loc[gm, gt] = p
        n_grid.loc[gm, gt]  = n

In [ ]:
# ---- Plateau & anti-cliff ----
def plateau_check(pf_grid, n_grid, chosen, pf_min=1.2, n_min=30, max_drop_pct=30.0):
    gm, gt = chosen
    r_idx = list(pf_grid.index).index(gm); c_idx = list(pf_grid.columns).index(gt)
    center = float(pf_grid.iloc[r_idx, c_idx])
    if not np.isfinite(center) or center < pf_min:
        return {"center_pf": center, "plateau": False, "worst_neighbor_drop_pct": np.nan,
                "anticliff": False, "passed": False, "pct_cells_pf_ok": float("nan"),
                "best_cell": (None, None), "best_pf": float("nan"), "edge_of_grid": True}
    drops = []
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == 0 and dc == 0: continue
            rr, cc = r_idx + dr, c_idx + dc
            if 0 <= rr < len(pf_grid.index) and 0 <= cc < len(pf_grid.columns):
                v = float(pf_grid.iloc[rr, cc])
                if np.isfinite(v) and v > 0:
                    drops.append((center - v) / center * 100)
    worst = max(drops) if drops else 0.0
    ok_nb = ((pf_grid.iloc[r_idx-1 if r_idx>0 else 0: r_idx+2, c_idx-1 if c_idx>0 else 0: c_idx+2] >= pf_min).all().all()
             and (n_grid.iloc[r_idx-1 if r_idx>0 else 0: r_idx+2, c_idx-1 if c_idx>0 else 0: c_idx+2] >= n_min).all().all())
    anticliff = worst <= max_drop_pct
    best_idx = np.unravel_index(np.nanargmax(pf_grid.values), pf_grid.shape)
    return {
        "center_pf": center,
        "plateau": bool(ok_nb),
        "worst_neighbor_drop_pct": float(worst),
        "anticliff": bool(anticliff),
        "passed": bool(ok_nb and anticliff),
        "pct_cells_pf_ok": float((pf_grid.values >= pf_min).mean() * 100),
        "best_cell": (pf_grid.index[best_idx[0]], pf_grid.columns[best_idx[1]]),
        "best_pf": float(np.nanmax(pf_grid.values)),
        "edge_of_grid": (r_idx == 0 or r_idx == len(pf_grid.index)-1 or c_idx == 0 or c_idx == len(pf_grid.columns)-1),
    }

CHOSEN = (float(GAP_MIN), float(GAP_THR))
chk = plateau_check(pf_grid, n_grid, CHOSEN)
print("=" * 70); print(f"PLATEAU DIAGNOSTIC — chosen ({GAP_MIN:g}, {GAP_THR:g})"); print("=" * 70)
for k, v in chk.items(): print(f"  {k:<28}: {v}")
print("=" * 70)

In [ ]:
# ---- Plot the grid ----
fig, ax = plt.subplots(figsize=(9, 5.5), facecolor="#15171c")
ax.set_facecolor("#1b1e26")
im = ax.imshow(pf_grid.values, cmap="Greens", vmin=1.0, vmax=max(3.0, np.nanmax(pf_grid.values) * 0.9), aspect="auto")
ax.set_xticks(range(len(GAP_THR_VALUES))); ax.set_xticklabels([f"{v:g}" for v in GAP_THR_VALUES], color="#9aa0ab")
ax.set_yticks(range(len(GAP_MIN_VALUES))); ax.set_yticklabels([f"{v:g}" for v in GAP_MIN_VALUES], color="#9aa0ab")
ax.set_xlabel("gap_threshold (%)", color="#9aa0ab"); ax.set_ylabel("gap_min_trade (%)", color="#9aa0ab")
ax.set_title(f"{TICKER} — In-sample PF grid (gap_min × gap_threshold)", color="#e8e9ec", fontweight="bold", loc="left", pad=10)
for i in range(len(GAP_MIN_VALUES)):
    for j in range(len(GAP_THR_VALUES)):
        v = pf_grid.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if v > 2.5 else "#132620")
ia, ib = list(GAP_MIN_VALUES).index(GAP_MIN), list(GAP_THR_VALUES).index(GAP_THR)
ax.add_patch(plt.Rectangle((ib - 0.5, ia - 0.5), 1, 1, fill=False, ec="#f5a524", lw=2.5))
plt.tight_layout(); plt.show()

In [ ]:
# ---- 4D scenario sweep (2 exit modes × 2 cost levels) ----
exit_modes = [("eod", "EOD exit"), ("target_or_eod", "Target or EOD")]
costs = [(5.0, "5 bps"), (10.0, "10 bps")]
combos = []
for em, em_lbl in exit_modes:
    for slip_val, slip_lbl in costs:
        sub_pf = pd.DataFrame(index=GAP_MIN_VALUES, columns=GAP_THR_VALUES, dtype=float)
        for gm in GAP_MIN_VALUES:
            for gt in GAP_THR_VALUES:
                t = eng.run_gap_backtest(intraday_is, daily_is,
                    gap_min_trade=gm, gap_threshold=gt, gap_max_tf=GAP_MAX_TF,
                    entry_tf_min=ENTRY_TF, mr_fill_target=MR_FILL, tf_target_mult=TF_MULT,
                    mr_stop_atr_mult=MR_STOP, tf_stop_atr_mult=TF_STOP,
                    time_exit=em, skip_fridays=SKIP_FRI, slip_bps_per_side=slip_val)
                sub_pf.loc[gm, gt] = pf(t)
                combos.append({"exit": em_lbl, "cost": slip_lbl, "gap_min": gm, "gap_thr": gt, "pf": pf(t), "n": len(t)})

dfc = pd.DataFrame(combos)
pfs = dfc["pf"].values
pct_pf12 = float((pfs >= 1.2).mean() * 100); pct_pf10 = float((pfs >= 1.0).mean() * 100)
min_pf, med_pf, max_pf = float(np.nanmin(pfs)), float(np.nanmedian(pfs)), float(np.nanmax(pfs))
chosen_pf = dfc.loc[(dfc.exit=="EOD exit") & (dfc.cost=="5 bps") & (dfc.gap_min==GAP_MIN) & (dfc.gap_thr==GAP_THR), "pf"].values[0]
pctile_chosen = float((pfs < chosen_pf).mean() * 100)

print("=" * 70); print(f"4D SCENARIO SWEEP ({len(dfc)} combinations)"); print("=" * 70)
print(f"Combinations with PF ≥ 1.2: {pct_pf12:.1f}% ({int((pfs >= 1.2).sum())}/{len(pfs)})")
print(f"Combinations with PF ≥ 1.0: {pct_pf10:.1f}% ({int((pfs >= 1.0).sum())}/{len(pfs)})")
print(f"min / median / max PF: {min_pf:.2f} / {med_pf:.2f} / {max_pf:.2f}")
print(f"Chosen config PF ({GAP_MIN:g}, {GAP_THR:g}, EOD, 5 bps): {chosen_pf:.3f}  (percentile {pctile_chosen:.1f}%)")
print("=" * 70)

summ = (dfc.groupby(["exit", "cost"])["pf"]
        .agg(min="min", median="median", max="max", pct_ge_1_2=lambda x: (x >= 1.2).mean() * 100)
        .round(2))
print(summ.to_string())

In [ ]:
# ---- Conclusions & save ----
print("=" * 70); print("CONCLUSIONS"); print("=" * 70)
print(f"Chosen ({GAP_MIN:g}, {GAP_THR:g}): center PF={chk['center_pf']:.3f} | worst-neighbor drop={chk['worst_neighbor_drop_pct']:.1f}% | "
      f"plateau={'PASS' if chk['plateau'] else 'FAIL'} | anti-cliff={'PASS' if chk['anticliff'] else 'FAIL'}")
print(f"4D: {pct_pf12:.1f}% of combinations reach PF ≥ 1.2 (median {med_pf:.2f}).")

pd.DataFrame(pf_grid).to_csv(RUN_DIR / "02_pf_grid.csv")
pd.DataFrame(n_grid).to_csv(RUN_DIR / "02_n_grid.csv")
save_json(cfg, "02_optimization.json", {
    "variant": cfg["id"], "ticker": TICKER, "params_tested": cfg["params"],
    "is_end_date": str(is_end_date.date()), "n_is_sessions": int(len(daily_is)),
    "plateau_checks": chk,
    "grid_4d": {
        "n_combinations": int(len(dfc)),
        "pct_pf_ge_1_2": pct_pf12, "pct_pf_ge_1_0": pct_pf10,
        "min_pf": min_pf, "median_pf": med_pf, "max_pf": max_pf,
        "chosen_pf": float(chosen_pf), "chosen_percentile": pctile_chosen,
    },
})

In [ ]:
# ---- Optimization parity vs base ----
BASE_VARIANT = "base_nasdaq"
if BASE_VARIANT:
    try:
        cb = load_variant(BASE_VARIANT)
        print(f"Base variant loaded: {cb['id']}")
    except Exception as e:
        print(f"Base variant not comparable for gap edge: {e}")